In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "ENTER API KEY"

# Explicitly uninstall and then install to try and resolve potential dependency conflicts
!pip uninstall -y langchain langchain-openai langgraph langchain-community langsmith langchain-core
!pip install langchain langchain-openai langgraph

from langchain.tools import tool
from langchain_openai.chat_models import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from langgraph.graph import StateGraph
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages

import warnings
from warnings import filterwarnings
filterwarnings('ignore')

Found existing installation: langchain 1.2.10
Uninstalling langchain-1.2.10:
  Successfully uninstalled langchain-1.2.10
Found existing installation: langgraph 1.0.9
Uninstalling langgraph-1.0.9:
  Successfully uninstalled langgraph-1.0.9
Found existing installation: langsmith 0.7.6
Uninstalling langsmith-0.7.6:
  Successfully uninstalled langsmith-0.7.6
Found existing installation: langchain-core 1.2.15
Uninstalling langchain-core-1.2.15:
  Successfully uninstalled langchain-core-1.2.15
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.7/111.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.9/160.9 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.2/502.2 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 344.0/344.0 kB 26.1 MB/s eta 0:00:00


In [ ]:
# -- 1) Define our single business tool
@tool
def cancel_request(ticket_id: int) -> str:
  """Cancel a ticket that is a duplicate or a nonrelevant request."""
  #(Here you would call your real backend API)
  return f"Ticket {ticket_id} has been cancelled/"

# Define the graph state for better type hinting
class AgentState(TypedDict):
    ticket: dict
    messages: Annotated[list, add_messages]

# Initialize LLM once globally without tools
basic_llm = ChatOpenAI(model='gpt-4', temperature=0.0)
# Bind tools separately to avoid serialization issues during graph compilation
llm = basic_llm.bind_tools([cancel_request])

# -- 2) The agent "brain": invoke LLM, run tool, then invoke LLM again
def call_model(state: AgentState):
  msgs = state["messages"]
  ticket = state.get("ticket", {"ticket_id": "UNKNOWN"})

  #System prompt tells the model exactly what to do
  prompt = (
      f'''You are an access support agent.
      TICKET ID: {ticket['ticket_id']}
      If the requestor asks to cancel, call cancel_request(ticket_id)
      and then send a simple confirmation.
      Otherwise, just provide a normal response.'''
  )
  full = [SystemMessage(prompt)] + msgs

  # 1st LLM pass: decides whether to call our tool
  first_response = llm.invoke(full)
  out = [first_response]

  if getattr(first_response, "tool_calls", None):
    # Run cancel_request tool
    tc = first_response.tool_calls[0]
    result = cancel_request.invoke(tc["args"]) # Corrected: use invoke() method
    out.append(ToolMessage(content=result, tool_call_id=tc["id"]))

    # 2nd LLM pass: generate the final confirmation text
    second_response = llm.invoke(full + out)
    out.append(second_response)

  return {"messages": out} # StateGraph nodes should return a dictionary to update state

# Wire it all up in a Stategraph
def construct_graph():
  g = StateGraph(AgentState)
  g.add_node("assistant", call_model)
  g.set_entry_point("assistant") # Corrected entry point string
  return g.compile()

graph = construct_graph() # Define graph in global scope, outside if __name__ == "__main__"

In [ ]:
#Minimal evaluation check
example_ticket ={"ticket_id": 3489}
convo =[HumanMessage(content= '''Please cancel ticket #3489.
I already submitted my request.
This is a duplicate request.''')]
result = graph.invoke({"ticket": example_ticket, "messages": convo})

assert any("Ticket" in str(m.content) for m in result["messages"]), "Cancel ticket tool not called"
assert any("cancelled" in m.content.lower() for m in result["messages"]), "Confirmation message missing"
print("Agent passed minimal evaluation.")

Agent passed minimal evaluation.
